In [2]:
import numpy as np
import pandas as pd

# -----------------------------
# Configuration
# -----------------------------
rng = np.random.default_rng(seed=42)

machine_numbers = [f"M{number}" for number in range(1201, 1206)]
start_date = "2024-01-01"
end_date = "2025-12-31"

# Two-day event with a slight increasing trend
event_dates = ["2024-07-15", "2024-07-16"]
event_date_set = set(pd.to_datetime(event_dates))

locations = ["P1", "P2", "P3", "P4"]
directions = ["V", "H", "A"]

vibration_min = 3.0
vibration_max = 10.5
temperature_min = 30.0
temperature_max = 65.0
target_missing_fraction = 0.08

# -----------------------------
# Set high-day probability
# -----------------------------
dates = pd.date_range(start=start_date, end=end_date, freq="D")

total_machine_days = len(machine_numbers) * len(dates)
event_machine_days = len(machine_numbers) * len(event_dates)

# Target about 5% of machine-days as high-vibration days,
# including the two configured event days.
target_high_machine_days = 0.05 * total_machine_days
non_event_machine_days = total_machine_days - event_machine_days

high_day_probability = (
    target_high_machine_days - event_machine_days
) / non_event_machine_days

# -----------------------------
# Generate daily records
# -----------------------------
records = []

for machine_index, machine_number in enumerate(machine_numbers):
    machine_temperature_offset = rng.uniform(-1.0, 1.0)

    for day_index, timestamp in enumerate(dates):
        is_event_day = timestamp in event_date_set

        # Decide once per machine-day. If high, all four locations
        # will receive high vibration readings.
        is_high_day = is_event_day or (
            rng.random() < high_day_probability
        )

        row = {
            "Machine_number": machine_number,
            "time stamp": timestamp,
            "_high_vibration_day": is_high_day,  # helper for missing-value step
        }

        if is_event_day:
            event_day_number = event_dates.index(
                timestamp.strftime("%Y-%m-%d")
            )
            # Second event day is slightly higher than the first.
            high_base = 7.4 + 0.35 * event_day_number
        else:
            # Occasional high days are around 7 mm/s.
            high_base = rng.uniform(7.0, 7.5)

        for location_index, location in enumerate(locations):
            if is_high_day:
                location_base = high_base
            else:
                # Most normal readings remain close to 3.0 mm/s.
                location_base = rng.normal(loc=3.25, scale=0.18)

            # Bounded offsets keep the H, V, and A readings
            # at a point within 0.4 mm/s of one another.
            direction_offsets = rng.uniform(-0.2, 0.2, size=3)
            vibration_values = location_base + direction_offsets
            vibration_values = np.clip(
                vibration_values, vibration_min, vibration_max
            )

            for direction, value in zip(directions, vibration_values):
                row[f"{location}-{direction}"] = round(float(value), 2)

            # Temperature: mild seasonal variation and daily noise
            seasonal_variation = 3.0 * np.sin(
                2 * np.pi * day_index / 365.25
            )
            location_temperature_offset = location_index * 0.5

            temperature = (
                42.0
                + machine_temperature_offset
                + location_temperature_offset
                + seasonal_variation
                + rng.normal(0, 1.0)
            )
            temperature = np.clip(
                temperature, temperature_min, temperature_max
            )
            row[f"{location}-T"] = round(float(temperature), 2)

        records.append(row)

dataset = pd.DataFrame(records)

# Arrange columns in the requested order
columns = ["Machine_number", "time stamp"]
for location in locations:
    columns.extend([
        f"{location}-V",
        f"{location}-H",
        f"{location}-A",
        f"{location}-T",
    ])

# Keep the high-day flag temporarily for the missing-value step
dataset = dataset[columns + ["_high_vibration_day"]]

# -----------------------------
# Add approximately 8% missing values
# -----------------------------
# Do not remove vibration values from high-vibration days, so the
# coordinated high readings remain visible at all four locations.
vibration_columns = [
    f"{location}-{direction}"
    for location in locations
    for direction in directions
]

protected_cells = pd.DataFrame(
    False, index=dataset.index, columns=dataset.columns
)
high_day_rows = dataset["_high_vibration_day"]
protected_cells.loc[high_day_rows, vibration_columns] = True

eligible_cells = np.argwhere(~protected_cells.to_numpy())
number_to_remove = round(dataset.shape[0] * len(columns)
                         * target_missing_fraction)

chosen_positions = rng.choice(
    len(eligible_cells),
    size=number_to_remove,
    replace=False,
)

for position in eligible_cells[chosen_positions]:
    row_index, column_index = position
    dataset.iat[row_index, column_index] = np.nan

# Remove the temporary helper column from the final dataset
dataset = dataset[columns]

# -----------------------------
# Save dataset
# -----------------------------
output_file = "pump_motor_vibration_dataset_2024_2025.csv"
dataset.to_csv(output_file, index=False)

print(f"Dataset saved to: {output_file}")
print(f"Rows: {len(dataset)}")
print(f"Columns: {len(dataset.columns)}")
print(
    "Actual missing-value percentage: "
    f"{dataset.isna().sum().sum() / dataset.size * 100:.2f}%"
)
print(dataset.head())

C:\Users\deepak.rao\AppData\Local\Temp\ipykernel_36044\1942752212.py:158: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  dataset.iat[row_index, column_index] = np.nan


Dataset saved to: pump_motor_vibration_dataset_2024_2025.csv
Rows: 3655
Columns: 18
Actual missing-value percentage: 7.53%
  Machine_number time stamp  P1-V  P1-H  P1-A   P1-T  P2-V  P2-H  P2-A   P2-T  \
0          M1201 2024-01-01  3.26  3.61  3.52  42.23  3.23  3.20  3.42  43.11   
1          M1201        NaT  3.17  3.00   NaN  43.03  3.82   NaN  3.58    NaN   
2          M1201 2024-01-03  3.36  3.15  3.17  42.72  3.37  3.38  3.41  42.68   
3          M1201 2024-01-04  3.05  3.13  3.31  41.57  3.00  3.00  3.15  42.78   
4          M1201 2024-01-05  3.31  3.28  3.25  42.67  3.00  3.00  3.13  43.65   

   P3-V  P3-H  P3-A   P3-T  P4-V  P4-H  P4-A   P4-T  
0  3.43  3.34  3.47    NaN  3.46  3.51  3.35  45.27  
1  3.31  3.44   NaN  44.25   NaN  3.46  3.31  43.43  
2  3.00  3.00  3.20  44.62  3.05  3.00  3.00  44.86  
3  3.34   NaN  3.40  43.04  3.02  3.27  3.17  43.73  
4  3.00  3.27  3.20  44.49   NaN  3.00  3.24  45.09  
